# Hungary: a reproducible monthly research dataset

Two investor perspectives, **USD and EUR**, using publicly downloadable data and no API key.
The working version freezes downloads made on **13 November 2026**. A date cap alone does not
freeze later revisions, so the raw responses, SHA-256 checksums, and retrieval times are retained locally.
Run all cells in order. The last cell displays coverage and descriptive statistics.

This prepares data; it does not estimate the asset-pricing model. The dividend-yield predictor is omitted.
The world proxy is the **Fama–French Developed market portfolio**, preserving a longer history than an ETF.

**Environment:** use your existing Miniconda **base** kernel. The first code cell records package versions. If statsmodels needs updating, run `%pip install --upgrade statsmodels==0.15.0` in the current kernel, then **Restart Kernel and Run All**. No custom kernel is required.

In [1]:
# Adjust these settings. Latest runs never overwrite the fixed working snapshot.
START_MONTH = "2000-01"
FIXED_END_DATE = "2026-11-13"
USE_LATEST = False
MAX_US_INFLATION_CARRY_MONTHS = 1  # 0 = strict; never carry a missing inflation report.

from datetime import datetime, timezone
from zoneinfo import ZoneInfo
from pathlib import Path
from io import BytesIO, StringIO
import hashlib
import importlib.metadata
import json
import platform
import sys
import re
import zipfile
from urllib.parse import parse_qsl, urlencode, urlsplit, urlunsplit

ROOT = Path.cwd()  # Start Jupyter in the repository root.
# Use the currently selected kernel (including Miniconda base); record its versions.
runtime_versions = {p: importlib.metadata.version(p) for p in ['numpy', 'pandas', 'requests']}
print("Python:", sys.executable)
print("Packages:", runtime_versions)

import numpy as np
import pandas as pd
import requests
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry
from IPython.display import display

now = datetime.now(ZoneInfo("Europe/Vienna"))
end_date = pd.Timestamp(now.date() if USE_LATEST else FIXED_END_DATE)
start_month = pd.Period(START_MONTH, freq="M")
last_month = end_date.to_period("M") - 1  # Always exclude the endpoint's unfinished month.
fetch_from = (start_month - 24).start_time.strftime("%Y-%m-%d")
if start_month > last_month or start_month < pd.Period("2000-01", freq="M"):
    raise ValueError("Choose START_MONTH from 2000-01 through the last completed month.")
tag = "latest_" + now.strftime("%Y%m%dT%H%M%S%f") if USE_LATEST else "fixed_" + FIXED_END_DATE
RUN_DIR = ROOT / "data" / tag / ("from_" + START_MONTH)
RAW_DIR = RUN_DIR / "raw"
manifest_path = RUN_DIR / "manifest.json"
if not USE_LATEST and not manifest_path.exists() and now.date() != end_date.date():
    raise RuntimeError("Restore the original fixed snapshot, or set USE_LATEST=True. A later download cannot recreate the original vintage.")
RAW_DIR.mkdir(parents=True, exist_ok=True)
manifest = json.loads(manifest_path.read_text()) if manifest_path.exists() else {
    "recipe_version": 1, "mode": "latest" if USE_LATEST else "fixed",
    "requested_end_date": str(end_date.date()), "start_month": START_MONTH,
    "created_utc": datetime.now(timezone.utc).isoformat(), "sources": {}, "sealed": False,
}
if manifest["recipe_version"] != 1:
    raise ValueError("Snapshot recipe version differs from this notebook.")
pd.set_option("display.max_rows", 40)
pd.set_option("display.max_columns", 12)
print(f"Mode: {manifest['mode']} | endpoint: {end_date.date()} | calendar cap: {last_month}")
print(f"Snapshot: {RUN_DIR.relative_to(ROOT)}")

Python: /Users/lajosgalambos/miniconda3/bin/python
Packages: {'numpy': '2.4.4', 'pandas': '3.0.2', 'requests': '2.32.5'}


Mode: fixed | endpoint: 2026-11-13 | calendar cap: 2026-10
Snapshot: data/fixed_2026-11-13/from_2000-01


## Sources and conventions

| Input | Public source | Convention |
|---|---|---|
| Hungarian market | [Budapest Stock Exchange](https://www.bse.hu/Prices-and-Markets/Data-download) | BUX closing levels in HUF; BUX already includes dividends |
| Developed world / US markets and USD cash | [French data library](https://mba.tuck.dartmouth.edu/pages/faculty/ken.french/data_library.html) | Reconstruct simple market returns as `Mkt-RF + RF`, then take `log(1 + return)` |
| HUF per EUR; USD per EUR | [ECB](https://data.ecb.europa.eu/help/api/data-examples) | Daily reference rates, last available observation of each month |
| EUR short rate | [ECB one-month Euribor](https://data.ecb.europa.eu/search-results?searchTerm=euribor) | Previous month's average annual rate, ACT/360, used as a **cash proxy** |
| US yields and CPI | [FRED](https://fred.stlouisfed.org/) | `GS10`, `TB3MS`, `BAA`, `AAA`, `CPIAUCNS`; public CSV, no key |
| Hungarian / euro-area HICP | [Eurostat](https://ec.europa.eu/eurostat/databrowser/view/prc_hicp_minr/default/table?lang=en) | All-items HICP, 2025=100; euro-area changing-composition aggregate |

EUR cash is an adaptation: Euribor contains bank credit risk, and the previous month's average is
an approximation to a rate locked at the start of the investment month. USD cash uses French's
one-month Treasury-bill return. Do not interpret their difference as a pure currency effect.

Market and FX observations are aligned by calendar month, with each market's own closing date;
ECB reference fixings are not simultaneous with equity closes. All financial returns are
**monthly log returns in decimals**. Interest-rate predictors are in annual percentage points.
Inflation uses US CPI and European HICP, which are not identical price-index concepts.

US CPI has no October 2025 observation because of the
[2025 government shutdown](https://www.bls.gov/cpi/additional-resources/2025-federal-government-shutdown-impact-cpi-faq.htm).
Keep that raw gap. For the conditioning variable only, carry the last observed **annual inflation rate**
for at most `MAX_US_INFLATION_CARRY_MONTHS` months and flag it. No CPI index level, asset return,
exchange rate, or price is interpolated or carried forward. Set the limit to zero for strict handling;
the complete-sample check will then identify the affected internal gap.

The fixed snapshot captures today's downloaded vintage, not a full historical real-time vintage.
Retain the entire `data/fixed_2026-11-13/` folder for later exact reproduction. Raw data stay local;
public download access does not imply unrestricted redistribution.

In [2]:
# Cache raw bytes, record provenance, and fail if a frozen file changes.
session = requests.Session()
# This standard client header also works with FRED's public CSV endpoint.
session.headers["User-Agent"] = "curl/8.7.1"
retry = Retry(total=3, backoff_factor=1, status_forcelist=[429, 500, 502, 503, 504],
              allowed_methods=["GET", "POST"])
session.mount("https://", HTTPAdapter(max_retries=retry))

def sha256(data):
    return hashlib.sha256(data).hexdigest()

def save_manifest():
    temporary = manifest_path.with_suffix(".tmp")
    temporary.write_text(json.dumps(manifest, indent=2, sort_keys=True) + "\n")
    temporary.replace(manifest_path)

def fetch(name, url, *, params=None, payload=None):
    path = RAW_DIR / name
    if path.exists():
        data = path.read_bytes()
        record = manifest["sources"].get(name)
        if record is None or sha256(data) != record["sha256"]:
            raise ValueError(f"Snapshot checksum mismatch: {name}. Restore the saved original.")
        return data
    if manifest["sealed"]:
        raise FileNotFoundError(f"Frozen source missing: {path}. Restore the snapshot; no re-fetch is allowed.")
    print(f"Downloading {name} ...", flush=True)
    response = session.request("POST" if payload is not None else "GET", url,
                               params=params, json=payload, timeout=(10, 45))
    response.raise_for_status()
    data = response.content
    if not data or data.lstrip().lower().startswith((b"<!doctype html", b"<html")):
        raise ValueError(f"Expected data, received an empty response or HTML: {name}")
    # CSRF is only an ephemeral public-form token; never retain it in provenance.
    parts = urlsplit(response.url)
    safe_query = urlencode([(k, v) for k, v in parse_qsl(parts.query) if k != "_csrf"])
    safe_url = urlunsplit((parts.scheme, parts.netloc, parts.path, safe_query, ""))
    path.with_suffix(path.suffix + ".tmp").write_bytes(data)
    path.with_suffix(path.suffix + ".tmp").replace(path)
    manifest["sources"][name] = {
        "url": safe_url, "method": response.request.method, "payload": payload,
        "retrieved_utc": datetime.now(timezone.utc).isoformat(),
        "bytes": len(data), "sha256": sha256(data),
    }
    save_manifest()
    return data

def monthly_values(dates, values, name):
    series = pd.Series(np.asarray(values, dtype=float), index=pd.to_datetime(dates), name=name)
    if series.index.has_duplicates:
        raise ValueError(f"Duplicate observation dates: {name}")
    return series.sort_index().resample("ME").last().to_period("M")

def french_market(raw):
    with zipfile.ZipFile(BytesIO(raw)) as archive:
        text = archive.read(archive.namelist()[0]).decode("utf-8-sig")
    # Monthly rows have six-digit dates; ignore prose and the annual-return section.
    rows = [line for line in text.splitlines() if re.match(r"^\s*\d{6}\s*,", line)]
    frame = pd.read_csv(StringIO("\n".join(rows)), header=None,
                        names=["month", "Mkt-RF", "SMB", "HML", "RF"])
    frame.index = pd.PeriodIndex(frame.pop("month").astype(str).str.strip(), freq="M")
    if frame.index.has_duplicates or frame.empty:
        raise ValueError("Invalid French monthly return file.")
    frame = frame.replace([-99.99, -999], np.nan)
    return pd.DataFrame({"market": (frame["Mkt-RF"] + frame["RF"]) / 100,
                         "rf": frame["RF"] / 100})

In [3]:
# 1. Public monthly market portfolios. No ETF / paid index history is required.
ff_root = "https://mba.tuck.dartmouth.edu/pages/faculty/ken.french/ftp/"
world = french_market(fetch("world_factors.zip", ff_root + "Developed_3_Factors_CSV.zip"))
us = french_market(fetch("us_factors.zip", ff_root + "F-F_Research_Data_Factors_CSV.zip"))
print(f"World market available through {world.index.max()}; US through {us.index.max()}.")

World market available through 2026-09; US through 2026-09.


In [4]:
# 2. Official BUX CSV downloads, in intervals within the public form's five-year limit.
bse_base = "https://www.bse.hu"
bse_page = bse_base + "/Prices-and-Markets/Data-download"
chunks = []
for year in range(pd.Timestamp(fetch_from).year, end_date.year + 1, 5):
    first = pd.Timestamp(year=year, month=1, day=1)
    last = min(pd.Timestamp(year=year + 4, month=12, day=31), end_date)
    chunks.append((first, last, f"bux_{first:%Y%m%d}_{last:%Y%m%d}.csv"))

if any(not (RAW_DIR / name).exists() for _, _, name in chunks) and not manifest["sealed"]:
    response = session.get(bse_page, timeout=(10, 45))
    response.raise_for_status()
    page = response.text
    paths = dict(re.findall(r'id="(\w+Url)"\s+[\w-]+="([^"]+)"', page))
    csrf = re.search(r'name="_csrf"\s+content="([^"]+)"', page).group(1)
    generator_url = bse_base + paths["historicalGeneratorUrl"]
    instruments = session.get(bse_base + paths["instrumentUrl"], params={
        "marketType": "index", "groupId": "-1", "resolution": "DAY_TO_DAY",
        "startDate": chunks[0][0].strftime("%Y.%m.%d."),
        "endDate": chunks[0][1].strftime("%Y.%m.%d."),
    }, timeout=(10, 45))
    instruments.raise_for_status()
    bux_id = next(item["id"] for item in instruments.json() if item["code"] == "BUX")

bux_parts = []
for first, last, name in chunks:
    if (RAW_DIR / name).exists() or manifest["sealed"]:
        raw = fetch(name, bse_page)  # Offline: the saved provenance identifies the original URL.
    else:
        payload = {
            "startingValue": first.strftime("%Y.%m.%d."), "endingValue": last.strftime("%Y.%m.%d."),
            "resolution": "DAY_TO_DAY", "market": "INDEX", "format": "CSV", "type": "CLOSING",
            "currentCategory": "-1",
            "selectionList": [{"category": "Indexek", "selectedInstruments": [{"id": bux_id, "code": "BUX"}]}],
        }
        raw = fetch(name, generator_url, params={"_csrf": csrf}, payload=payload)
    part = pd.read_csv(BytesIO(raw), encoding="utf-8-sig")
    if not {"Name", "Date", "Last price"}.issubset(part.columns) or not part["Name"].eq("BUX").all():
        raise ValueError(f"Unexpected BSE CSV format: {name}")
    part["Date"] = pd.to_datetime(part["Date"], format="%d.%m.%Y.")
    part["Last price"] = pd.to_numeric(part["Last price"], errors="raise")
    bux_parts.append(part.dropna(subset=["Last price"]))
bux_daily = pd.concat(bux_parts).sort_values("Date")
bux = monthly_values(bux_daily["Date"], bux_daily["Last price"], "bux_huf")
print(f"BUX: {len(bux_daily):,} daily closes, {bux.index.min()} to {bux.index.max()}.")

BUX: 7,205 daily closes, 1998-01 to 2026-11.


In [5]:
# 3. ECB reference exchange rates and one-month Euribor.
ecb = "https://data-api.ecb.europa.eu/service/data/"
fx_raw = fetch("ecb_fx.csv", ecb + "EXR/D.HUF+USD.EUR.SP00.A", params={
    "startPeriod": fetch_from, "endPeriod": str(end_date.date()), "format": "csvdata"})
fx_daily = pd.read_csv(BytesIO(fx_raw), usecols=["TIME_PERIOD", "CURRENCY", "OBS_VALUE"])
if fx_daily.duplicated(["TIME_PERIOD", "CURRENCY"]).any():
    raise ValueError("Duplicate ECB FX observations.")
fx_daily = fx_daily.pivot(index="TIME_PERIOD", columns="CURRENCY", values="OBS_VALUE")
fx = pd.concat([
    monthly_values(fx_daily.index, fx_daily["HUF"], "huf_per_eur"),
    monthly_values(fx_daily.index, fx_daily["USD"], "usd_per_eur"),
], axis=1)
euribor_raw = fetch("ecb_euribor_1m.csv", ecb + "FM/M.U2.EUR.RT.MM.EURIBOR1MD_.HSTA", params={
    "startPeriod": fetch_from[:7], "endPeriod": str(last_month), "format": "csvdata"})
euribor_frame = pd.read_csv(BytesIO(euribor_raw), usecols=["TIME_PERIOD", "OBS_VALUE"])
euribor = monthly_values(euribor_frame["TIME_PERIOD"], euribor_frame["OBS_VALUE"], "euribor_1m_pa_pct")

In [6]:
# 4. Public FRED CSV and Eurostat HICP. No credentials are used or stored.
fred_raw = fetch("fred.csv", "https://fred.stlouisfed.org/graph/fredgraph.csv", params={
    "id": "GS10,TB3MS,BAA,AAA,CPIAUCNS", "cosd": fetch_from, "coed": str(end_date.date())})
fred = pd.read_csv(BytesIO(fred_raw), index_col=0, parse_dates=True, na_values=".")
if fred.index.has_duplicates or not {"GS10", "TB3MS", "BAA", "AAA", "CPIAUCNS"}.issubset(fred.columns):
    raise ValueError("Unexpected FRED CSV format.")
fred = fred.apply(pd.to_numeric, errors="raise").resample("ME").last().to_period("M")

hicp_raw = fetch("eurostat_hicp.json", "https://ec.europa.eu/eurostat/api/dissemination/statistics/1.0/data/prc_hicp_minr",
    params={"lang": "en", "freq": "M", "coicop18": "TOTAL", "unit": "I25",
            "geo": ["HU", "EA"], "sinceTimePeriod": fetch_from[:7]})
hicp_json = json.loads(hicp_raw)
expected_dimensions = ["freq", "unit", "coicop18", "geo", "time"]
if hicp_json["id"] != expected_dimensions or hicp_json["size"][:3] != [1, 1, 1]:
    raise ValueError("Eurostat dimensions changed; inspect the response before continuing.")
shape = hicp_json["size"]
values = np.full(int(np.prod(shape)), np.nan)
for position, value in hicp_json["value"].items():
    values[int(position)] = value
values = values.reshape(shape)[0, 0, 0]
categories = hicp_json["dimension"]
dates = sorted(categories["time"]["category"]["index"], key=categories["time"]["category"]["index"].get)
hicp = pd.DataFrame({
    "hicp_hu": values[categories["geo"]["category"]["index"]["HU"]],
    "hicp_ea": values[categories["geo"]["category"]["index"]["EA"]],
}, index=pd.PeriodIndex(dates, freq="M"))

# Reindex BEFORE taking differences/lags, so a missing month cannot become a multi-month return.
grid = pd.period_range(fetch_from[:7], last_month, freq="M")
inputs = pd.concat([bux, fx, world["market"].rename("world_simple_usd"),
                   us["market"].rename("us_simple_usd"), us["rf"].rename("rf_simple_usd"),
                   euribor, fred, hicp], axis=1).reindex(grid)
inputs.index.name = "month"
positive = ["bux_huf", "huf_per_eur", "usd_per_eur", "CPIAUCNS", "hicp_hu", "hicp_ea"]
if (inputs[positive].dropna() <= 0).any().any():
    raise ValueError("A price, exchange rate, or CPI index is non-positive.")
if (inputs[["world_simple_usd", "us_simple_usd", "rf_simple_usd"]].dropna() <= -1).any().any():
    raise ValueError("A simple return is <= -100%.")

## Build both investor perspectives

For `S = HUF per investor-currency unit`, the unhedged Hungarian log return is
`diff(log(BUX)) - diff(log(S))`. Convert the **same** world and US portfolios into EUR.
Subtract the matching monthly log cash return only after currency conversion.

Global predictors: lagged changes in the US 3-month bill rate and 10-year-minus-3-month term spread,
plus the lagged Baa-minus-Aaa credit spread. The bill rate substitutes for the paper's Eurodollar rate.
Inflation gaps are **investor-country inflation minus Hungarian inflation**, lagged two months to
allow for publication. Rate predictors use one-month lags. These lags are transparent approximations;
the downloaded histories are not vintage/release-calendar data.

The floating-regime indicator first equals one in March 2008, the first full month after the
[26 February 2008 regime change](https://www.mnb.hu/en/monetary-policy/monetary-policy-framework/exchange-rate-regime),
which was known at the start of March. It is a conditioning variable, not a trading signal.

In [7]:
data = pd.DataFrame(index=grid)
huf_per_usd = inputs["huf_per_eur"] / inputs["usd_per_eur"]
eur_usd_change = np.log(inputs["usd_per_eur"]).diff()
data["fx_huf_usd"] = -np.log(huf_per_usd).diff()  # Positive = HUF appreciation.
data["fx_huf_eur"] = -np.log(inputs["huf_per_eur"]).diff()
data["fx_eur_usd"] = eur_usd_change  # Positive = EUR appreciation against USD.
hu_local = np.log(inputs["bux_huf"]).diff()
data["r_hu_usd"] = hu_local + data["fx_huf_usd"]
data["r_hu_eur"] = hu_local + data["fx_huf_eur"]
for market in ["world", "us"]:
    data[f"r_{market}_usd"] = np.log1p(inputs[f"{market}_simple_usd"])
    data[f"r_{market}_eur"] = data[f"r_{market}_usd"] - eur_usd_change
data["rf_usd"] = np.log1p(inputs["rf_simple_usd"])
data["rf_eur_proxy"] = np.log1p(inputs["euribor_1m_pa_pct"].shift(1) / 100 * grid.days_in_month / 360)
for currency, cash in [("usd", "rf_usd"), ("eur", "rf_eur_proxy")]:
    for market in ["hu", "world", "us"]:
        data[f"rx_{market}_{currency}"] = data[f"r_{market}_{currency}"] - data[cash]

data["short_rate_change_lag1_pp"] = inputs["TB3MS"].diff().shift(1)
data["term_spread_change_lag1_pp"] = (inputs["GS10"] - inputs["TB3MS"]).diff().shift(1)
data["credit_spread_lag1_pp"] = (inputs["BAA"] - inputs["AAA"]).shift(1)
inflation = inputs[["CPIAUCNS", "hicp_hu", "hicp_ea"]].pct_change(12, fill_method=None) * 100
us_inflation = inflation["CPIAUCNS"]
us_inflation_for_z = (us_inflation.ffill(limit=MAX_US_INFLATION_CARRY_MONTHS)
                      if MAX_US_INFLATION_CARRY_MONTHS > 0 else us_inflation)
data["inflation_gap_usd_lag2_pp"] = (us_inflation_for_z - inflation["hicp_hu"]).shift(2)
data["inflation_gap_eur_lag2_pp"] = (inflation["hicp_ea"] - inflation["hicp_hu"]).shift(2)
data["flag_us_inflation_carried_lag2"] = (us_inflation.isna() & us_inflation_for_z.notna()).shift(2, fill_value=False).astype(int)
data["floating_regime_known"] = (grid >= pd.Period("2008-03", freq="M")).astype(int)
data = data.loc[start_month:last_month]
data.index.name = "month"

# Keep missing observations visible. Trim only the incomplete leading/trailing edges.
complete = data.notna().all(axis=1)
if not complete.any():
    raise ValueError("No complete common sample. Check source coverage below.")
first_month, final_month = data.index[complete][[0, -1]]
working = data.loc[first_month:final_month].copy()
if working.isna().any().any():
    raise ValueError("Internal missing data: inspect data[data.isna().any(axis=1)]. No interpolation is performed.")
if not np.isfinite(working.to_numpy()).all():
    raise ValueError("Non-finite observations in the working dataset.")

# Accounting checks: each asset has the same USD/EUR currency-conversion difference.
for market in ["hu", "world", "us"]:
    np.testing.assert_allclose(working[f"r_{market}_usd"] - working[f"r_{market}_eur"],
                               working["fx_eur_usd"], atol=1e-12, rtol=0)
np.testing.assert_allclose(working["fx_huf_usd"] - working["fx_huf_eur"],
                           working["fx_eur_usd"], atol=1e-12, rtol=0)
assert working.index.equals(pd.period_range(first_month, final_month, freq="M"))
print(f"Working sample: {first_month} to {final_month}, {len(working):,} complete months, {working.shape[1]} variables.")
print(f"Calendar cap: {last_month}; later incomplete months retained in monthly_panel.csv.")
print("Carried US inflation reports used:", working.index[working["flag_us_inflation_carried_lag2"].eq(1)].astype(str).tolist())

Working sample: 2000-01 to 2026-09, 321 complete months, 24 variables.
Calendar cap: 2026-10; later incomplete months retained in monthly_panel.csv.
Carried US inflation reports used: ['2025-12']


In [8]:
# Save deterministic exports and seal the working snapshot. Later fixed reruns need no internet.
selected_inputs = inputs.loc[start_month:last_month]
coverage = pd.DataFrame({
    "observations": selected_inputs.count(), "missing_months": selected_inputs.isna().sum(),
    "first": selected_inputs.apply(lambda s: str(s.first_valid_index())),
    "last": selected_inputs.apply(lambda s: str(s.last_valid_index())),
})
summary = pd.DataFrame(index=working.columns)
summary["unit"] = ["percentage points" if c.endswith("_pp") else
                   "indicator" if c == "floating_regime_known" or c.startswith("flag_") else "monthly log %" for c in working.columns]
summary["n"] = working.count()
summary["missing"] = working.isna().sum()
scaled = working.copy()
return_columns = summary.index[summary["unit"].eq("monthly log %")]
scaled[return_columns] *= 100  # Exports remain in decimals; only descriptive statistics are scaled.
for statistic in ["mean", "std", "min", "max"]:
    summary[statistic] = getattr(scaled, statistic)()

exports = {"monthly_inputs.csv": selected_inputs, "monthly_panel.csv": data,
           "dataset.csv": working, "source_coverage.csv": coverage, "summary.csv": summary}
for name, frame in exports.items():
    content = frame.to_csv(float_format="%.17g", lineterminator="\n").encode("utf-8")
    if manifest["sealed"]:
        expected = manifest["exports"][name]
        path = RUN_DIR / name
        if sha256(content) != expected or (path.exists() and sha256(path.read_bytes()) != expected):
            raise ValueError(f"Frozen export differs: {name}. Inspect changes; do not replace the working snapshot.")
    (RUN_DIR / name).write_bytes(content)
if not manifest["sealed"]:
    manifest.update({
        "sealed": True, "working_first_month": str(first_month), "working_last_month": str(final_month),
        "observations": len(working), "variables": list(working.columns),
        "max_us_inflation_carry_months": MAX_US_INFLATION_CARRY_MONTHS,
        "exports": {name: sha256((RUN_DIR / name).read_bytes()) for name in exports},
        "environment": {"python": platform.python_version(), **{
            package: importlib.metadata.version(package) for package in ["numpy", "pandas", "requests", "ipykernel"]}},
    })
    save_manifest()
print(f"Verified snapshot and exports: {RUN_DIR.relative_to(ROOT)}")

Verified snapshot and exports: data/fixed_2026-11-13/from_2000-01


## Dataset summary

`dataset.csv` is the **complete, common working sample** for both investor perspectives.
`monthly_panel.csv` also retains incomplete boundary months so source publication lags are visible.
The coverage table describes available inputs up to the calendar cap; the statistics use only the
complete working sample. Reported means and standard deviations are monthly, not annualised.

To check more recent data later, change only `USE_LATEST = True` and run all cells. A separate dated
folder is created. Reset it to `False` to return to the frozen working version.
If a download fails on the first run, rerun: successfully cached files are reused. After sealing,
missing/corrupt raw files cause a clear error rather than fetching a different vintage.

In [9]:
display(pd.DataFrame({
    "mode": [manifest["mode"]], "endpoint": [manifest["requested_end_date"]],
    "working_start": [str(first_month)], "working_end": [str(final_month)],
    "months": [len(working)], "variables": [working.shape[1]],
}))
display(coverage.rename_axis("input"))
display(summary.round(4).rename_axis("variable"))

,mode,endpoint,working_start,working_end,months,variables
0,fixed,2026-11-13,2000-01,2026-09,321,24


,observations,missing_months,first,last
input,,,,
bux_huf,322,0,2000-01,2026-10
huf_per_eur,322,0,2000-01,2026-10
usd_per_eur,322,0,2000-01,2026-10
world_simple_usd,321,1,2000-01,2026-09
us_simple_usd,321,1,2000-01,2026-09
rf_simple_usd,321,1,2000-01,2026-09
euribor_1m_pa_pct,322,0,2000-01,2026-10
GS10,322,0,2000-01,2026-10
TB3MS,322,0,2000-01,2026-10


,unit,n,missing,mean,std,min,max
variable,,,,,,,
fx_huf_usd,monthly log %,321,0,-0.0750,3.8537,-19.7003,11.5921
fx_huf_eur,monthly log %,321,0,-0.1131,2.1636,-11.4587,6.7965
fx_eur_usd,monthly log %,321,0,0.0382,2.6757,-11.4389,8.9385
r_hu_usd,monthly log %,321,0,0.7976,8.6153,-52.2597,23.4268
r_hu_eur,monthly log %,321,0,0.7595,7.4790,-40.8208,21.2394
r_world_usd,monthly log %,321,0,0.5825,4.5123,-21.6044,12.5310
r_world_eur,monthly log %,321,0,0.5443,4.0989,-14.4846,11.0619
r_us_usd,monthly log %,321,0,0.6787,4.5408,-18.7776,12.7513
r_us_eur,monthly log %,321,0,0.6405,4.4989,-14.0225,13.4842
